# --Silver Layer--


--This notebook reads Employee data from the Bronze layer, performs cleansing, standardization, validation, and writes the curated data to the Silver layer--

In [0]:
import os
import sys

# Current notebook path
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)
print(sys.path[:3])

In [0]:
from src.constants import *
from src.common_functions import *
from src.date_utils import *
from src.validations import *

--Read the bronze employee--

In [0]:
employee_df = spark.table(EMPLOYEE_BRONZE_TABLE)

display(employee_df)

In [0]:
print(employee_df.count())

-Trim and Clean-

In [0]:
employee_df = trim_columns(employee_df)

employee_df = replace_blank_with_null(employee_df)

employee_df = replace_nan_with_null(employee_df)

-Convert Dates-

In [0]:
date_columns = [
    "DOB",
    "Employee_Added",
    "Hire_Date",
    "Rehire_Date",
    "Termination_Date"
]

for column in date_columns:
    employee_df = convert_mixed_date(employee_df, column)

In [0]:
from pyspark.sql.functions import col, to_timestamp

employee_df = (
    employee_df
    .withColumn("Badge_#", col("Badge_#").cast("int"))
    .withColumn("Facility_Code", col("Facility_Code").cast("int"))
    .withColumn("Labor_Position_Code", col("Labor_Position_Code").cast("int"))
    .withColumn("ingestion_timestamp", to_timestamp("ingestion_timestamp"))
)

In [0]:
employee_df.printSchema()

--Remove Duplicates --

In [0]:
employee_df = employee_df.dropDuplicates(["Employee_Code"])

print(f"Rows after removing duplicates: {employee_df.count()}")

--Null Validation--

In [0]:
import importlib
import src.date_utils as date_utils

importlib.reload(date_utils)
from src.date_utils import *

In [0]:
print(employee_df.schema)

In [0]:
employee_df.printSchema()

## --Data Quality Validation--

In [0]:
print("Total Rows:", employee_df.count())

In [0]:
from pyspark.sql.functions import count

employee_df.groupBy("Employee_Code") \
    .agg(count("*").alias("cnt")) \
    .filter("cnt > 1") \
    .show()

In [0]:
employee_df.filter(
    employee_df.Employee_Code.isNull()
).show()

In [0]:
employee_df.filter(
    employee_df.FirstName.isNull()
).show()

In [0]:
employee_df.filter(
    employee_df.LastName.isNull()
).show()

## WaterMarkss

In [0]:
from src.watermark import get_watermark, update_watermark
from pyspark.sql import functions as F

print("Watermark framework imported")

In [0]:
pipeline_name = "Silver Employee"
source_name = "Employee_Payroll.xlsx"

last_watermark = get_watermark(
    pipeline_name,
    source_name
)

print("Last processed watermark:", last_watermark)

In [0]:
if last_watermark is None:

    employee_incremental_df = employee_df

else:

    employee_incremental_df = (
        employee_df
        .filter(
            F.col("ingestion_timestamp") > F.lit(last_watermark)
        )
    )

print(
    "Incremental Employee records:",
    employee_incremental_df.count()
)



### --Write to the silver--

In [0]:
# --Write to the silver--

incremental_count = employee_incremental_df.count()

if incremental_count == 0:

    print("No new Employee records to load.")
    print("Silver Employee table remains unchanged.")

else:

    employee_incremental_df.write \
        .mode("append") \
        .saveAsTable(EMPLOYEE_SILVER_TABLE)

    print(
        f"Incremental Employee records loaded: {incremental_count}"
    )

    # Update watermark only after successful Silver load
    new_watermark = (
        employee_incremental_df
        .agg(
            F.max("ingestion_timestamp")
            .alias("max_ingestion_timestamp")
        )
        .collect()[0]["max_ingestion_timestamp"]
    )

 
    update_watermark(
        pipeline_name,
        source_name,
        new_watermark
    )

    print(
        "Watermark updated to:",
        new_watermark
    )

In [0]:
print(
    "Silver Employee count:",
    spark.table(EMPLOYEE_SILVER_TABLE).count()
)

In [0]:
print(
    "Current watermark:",
    get_watermark(
        pipeline_name,
        source_name
    )
)

In [0]:
watermark_df = spark.table(
    "databricks_project1.silver.pipeline_watermark"
)

display(
    watermark_df.filter(
        F.col("pipeline_name") == "Silver Employee"
    )
)